# A2.2 · Bootstrapping the first credential

**Function A — Securing AI Architectures → Securing the Architecture — Identity and Ingress**  ·  *Security of AI*

Builds on **[A2.1 · Agent identity: user, workload, agent](https://spbreed.github.io/cyber-commons/lessons/A2.1.html)**.

| | |
|---|---|
| Tools used | SPIFFE/SPIRE |

## What this lesson is

**What it covers.** Exchange an attestation for a credential, then show a copied secret failing the same exchange.

**Why a security engineer needs it.** A pre-shared secret in an image or an environment variable is copyable, so possession stops being proof of identity. The control it builds is: platform attestation exchanged for a short-lived, workload-bound credential.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

An agent needs a credential to prove who it is, and it cannot be given one safely without already proving who it is. Every long-lived secret in your estate exists because somebody resolved that circle by giving up.

> **At CyberTravels.** Each of CyberTravels' four agents needs a credential to prove it is that agent, and cannot be handed one safely without already proving it. The long-lived bearer token in R5 exists because somebody resolved that circle by giving up.

## 2 · The framework

```
   to get a credential you must prove who you are
   to prove who you are you need a credential
                  |
             attestation breaks the circle
                  v
   platform says "this workload is what it claims"  (hardware/orchestrator)
                  |
                  v
   short-lived identity document ---> rotated automatically, never stored
```

**Mitigates: T9 Identity Spoofing & Impersonation.**

A2.1 says the workload needs its own identity. This lesson is about how it gets
one, because there is a circularity: to receive a credential securely the
workload must already prove who it is.

The wrong answer is a **pre-shared secret** — a key in the image, a token in an
environment variable, a file mounted at deploy time. All of them are copyable,
and a copyable secret makes possession the proof of identity. Anyone who reads
the image is the agent.

The control is **attestation**. The platform that started the workload already
knows things nobody else can forge: which image ran, in which namespace, under
which service account, on which node. It signs a statement to that effect, and
an identity service exchanges that statement for a short-lived credential bound
to that workload.

Three properties matter:

- **Non-copyable.** The attestation describes a running process. Copying the
  document to another machine produces a claim the platform will not sign.
- **Short-lived.** The credential expires in minutes, so theft has a deadline.
- **Bound.** It is issued *to* that workload identity, so presenting it from
  elsewhere fails.

This is what SPIFFE/SPIRE and every cloud workload-identity system do. The
lesson models the exchange, not the product.

> **What this control closes.**
>
> Makes **possession stop being proof**. Without it, A1.7 is unavoidable: a copyable secret means every holder is the agent.

## 3 · The check, as a skill

Whether CyberTravels' agents hold a credential or a secret is settled by four probes, not by reading the deployment manifest. The skill runs them: an unattested process, a genuine image nobody registered, a credential presented from another node, and one presented after its TTL.

### The skill — [`skills/identity/workload-attestation-check/SKILL.md`](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/identity/workload-attestation-check/SKILL.md)

```yaml
name: workload-attestation-check
description: >-
  Establish how a workload receives its first credential and whether possession
  of that credential is still proof of identity — testing an unattested process,
  an unregistered image, presentation from another node, and use after expiry.
  Use when reviewing bootstrap, SPIFFE/SPIRE, or any secret mounted at deploy
  time.
allowed-tools: Read, Grep, Glob
```

# Where the first credential comes from

There is a circularity at the start of every workload identity: to receive a
credential securely the workload must already prove who it is. A pre-shared
secret resolves it by making **possession** the proof, which means everyone who
can read the image is the agent. Attestation resolves it by having the platform
sign what only the platform knows.

## When to use this

Reviewing how any agent, job or pod authenticates for the first time. The
answer decides whether every later identity control rests on something or on a
copied file.

## Procedure

**1 — Find the first credential.** Trace back from a downstream call to where
the credential entered the process: an environment variable, a mounted file, a
secrets manager fetch, or an attestation exchange.

**2 — Ask who else can read it.** For a secret, that set is the set of people
who are currently the agent. Enumerate it — image layers, CI logs, anyone with
`exec` on the namespace, anyone who can read the manifest.

**3 — Test the unattested process.** A process that is not what the platform
started should receive nothing. If it receives a credential, the exchange is
authenticating a claim rather than a workload.

**4 — Test the unregistered-but-genuine image.** A real workload nobody
registered is the case people forget: attestation proves *what* is running, and
registration decides whether it should be. Both must refuse.

**5 — Test binding and lifetime.** Present a legitimately issued credential
from a different node, and again after its TTL. Both must fail. A credential
that travels is a secret with extra steps.

## Output contract

```json
{
  "source": "env|file|manager|attestation",
  "readable_by": ["str"],
  "probes": {"unattested": "issued|refused", "unregistered_image": "issued|refused",
             "wrong_node": "accepted|refused", "expired": "accepted|refused"},
  "properties": {"non_copyable": false, "short_lived": false, "bound": false},
  "ttl_seconds": 0
}
```

All three properties, or the credential is a secret: non-copyable because it
describes a running process, short-lived so theft has a deadline, bound so
presenting it elsewhere fails.

## Failure modes

- **Accepting a secrets manager as attestation.** It answers "who may fetch
  this", which is the same circularity one layer down.
- **Testing only the unattested case.** The unregistered genuine workload is
  the one that gets through.
- **Recording a long TTL as acceptable** because rotation exists. Rotation is
  not a deadline for a copy already taken.

In [ ]:
# The code is not in this notebook. It is this file in the repository:
#   https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/skills/identity/workload-attestation-check/scripts/workload_attestation_check.py
SCRIPT = "skills/identity/workload-attestation-check/scripts/workload_attestation_check.py"
REPO = "https://github.com/spbreed/cyber-commons"
BRANCH = "claude/vulnbench-setup-scheduling-81aqov"

import glob, os, subprocess, sys

CLONE = "/kaggle/working/cyber-commons"
_root = next((r for r in (".", "..", "../..", CLONE)
              if os.path.isfile(os.path.join(r, SCRIPT))), None)

if _root is None:
    # --filter=blob:none --sparse fetches the tree without the history or the
    # notebooks; `sparse-checkout set skills` then materialises only what runs.
    _c = subprocess.run(["git", "clone", "--depth", "1", "--filter=blob:none",
                         "--sparse", "--branch", BRANCH, REPO, CLONE],
                        capture_output=True, text=True)
    if _c.returncode:
        raise SystemExit(
            "could not fetch the skills: " + _c.stderr.strip()[-300:] +
            "\nOn Kaggle this needs Internet on in the notebook settings, which "
            "needs a phone-verified account. Without one, attach the dataset "
            "cybercommons/cyber-commons-skills instead — it holds the same tree.")
    subprocess.run(["git", "-C", CLONE, "sparse-checkout", "set", "skills"],
                   capture_output=True, text=True)
    _root = CLONE

_out = subprocess.run([sys.executable, os.path.join(_root, SCRIPT)],
                      capture_output=True, text=True,
                      env=dict(os.environ,
                               PYTHONPATH=os.path.join(_root, "skills/_runtime"),
                               PYTHONHASHSEED="0"))
print(_out.stdout, end="")
if _out.returncode:
    raise SystemExit(_out.stderr.strip()[-2000:])

## What you just proved

An unattested process receives no credential, a genuine but unregistered image receives none either, and a credential issued to a real workload is refused when presented from another node or after its five-minute expiry.

## Your turn

Find where one of your agents gets its first credential. If the answer is an environment variable or a mounted file, list everyone who can read it — that is the set of people who are currently that agent.

---

**Next → [A2.3 · Delegation that narrows, and survives audit](https://spbreed.github.io/cyber-commons/lessons/A2.3.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A2.2.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A2.2.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*